# 👤 EBLET-Lite: Procesamiento de Respuestas Reales

## 🎯 Objetivo
Procesar respuestas reales de participantes que han completado EBLET-Lite y generar:
- Perfil individual de bienestar laboral
- Cultura organizacional percibida
- Posición relativa vs benchmark (12,500 empleados)
- Recomendaciones personalizadas

## 📋 EBLET-Lite: Versión Abreviada de Cribado

EBLET-Lite es una **versión reducida** de EBLET Enterprise diseñada para aplicación rápida individual.

### Mapa de Equivalencias: Enterprise (72 preguntas) → Lite (23 preguntas)

| Dimensión | Enterprise | Lite | Estado |
|-----------|------------|------|--------|
| Contexto organizacional (JD-R) | Q1-Q15 | ❌ Eliminado | Demasiado amplio para versión rápida |
| Burnout emocional | Q16-Q22 | Q1-Q4 | ✅ Mantenido (screening) |
| Burnout cinismo | Q23-Q29 | Q3-Q4 | ✅ Parcialmente incluido |
| Eficacia profesional | Q30-Q36 | ❌ Eliminado | No esencial para cribado rápido |
| Boreout aburrimiento | Q37-Q44 | Q5-Q8 | ✅ Mantenido |
| Bienestar psicológico | Q45-Q49 | Q9-Q12 | ✅ Adaptado (WHO-5) |
| Satisfacción laboral | Q50-Q53 | ❌ Eliminado | Reflejado en rotación/bienestar |
| Autoeficacia | Q54-Q56 | ❌ Eliminado | No es núcleo del objetivo Lite |
| Rotación | Q57-Q59 | Q13-Q15 | ✅ Mantenido completo |
| Infraocupación | Q60-Q64 | Q8 | ✅ Simplificado |
| Cultura CVF | Q65-Q72 | Q16-Q23 | ✅ Mantenido completo |

### ⚠️ Limitaciones de EBLET-Lite

EBLET-Lite es una **versión abreviada de cribado individual** basada en las dimensiones centrales de EBLET Enterprise. Mantiene los constructos principales de bienestar laboral, pero reduce la profundidad diagnóstica para permitir una aplicación rápida (~8 minutos).

**Dimensiones no incluidas:**
- Recursos y demandas laborales (JD-R)
- Autoeficacia
- Satisfacción laboral completa
- Eficacia profesional
- Infraocupación completa

Por tanto, EBLET-Lite **evalúa indicadores relacionados con agotamiento y desconexión laboral**, pero no constituye una medición completa del burnout según MBI-GS.

##  Anonimato
Cada participante eligió un alias personal. Solo ellos pueden reconocerse en los resultados.

In [2]:

# IMPORTS Y CONFIGURACIÓN


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import os
import sys

RAIZ_PROYECTO = r"C:\Users\torre\OneDrive\Escritorio\EBLET-People-Analytics\Python"
sys.path.insert(0, os.path.join(RAIZ_PROYECTO, "src"))

# Paleta de colores para perfiles
color_perfil = {
    "🟢 Flourishing": "#2ecc71",
    "🟡 Estable Neutro": "#f1c40f",
    "🟠 Quemado Activo": "#e67e22",
    "🔵 Aburrido Crónico": "#3498db",
    " Crítico Dual": "#e74c3c",
    "⚫ Desvinculado": "#34495e"
}

print("✅ Notebook EBLET-Lite cargado correctamente")
print("📋 Versión: 23 preguntas (cribado individual)")
print("️  Nota: Esta es una versión abreviada de EBLET Enterprise (72 preguntas)")

✅ Notebook EBLET-Lite cargado correctamente
📋 Versión: 23 preguntas (cribado individual)
️  Nota: Esta es una versión abreviada de EBLET Enterprise (72 preguntas)


## 1.  Cargar Respuestas Reales de Google Forms

**Instrucciones:**
5. Guarda el archivo como: `respuestas_encuesta_individual.csv`
6. Muévelo a: `Python/datasets/respuestas_encuesta_individual.csv`

In [3]:
# =====================================================
# CARGAR CSV DE GOOGLE FORMS
# =====================================================

ruta_csv = os.path.join(RAIZ_PROYECTO, "datasets/respuestas_encuesta_individual.csv")

if not os.path.exists(ruta_csv):
    print("⚠️  El archivo no existe aún: " + ruta_csv)
    print("\n📋 Pasos para obtenerlo:")
    print("1. Ve a tu Google Forms")
    print("2. Pestaña 'Respuestas'")
    print("3. Clic en los 3 puntos verticales (⋮)")
    print("4. 'Descargar respuestas (.csv)'")
    print("5. Guarda el archivo como: respuestas_encuesta_individual.csv")
    print("6. Muévelo a: " + os.path.join(RAIZ_PROYECTO, "datasets/"))
    print("\n💡 Una vez tengas el archivo, re-ejecuta esta celda.")
    
    # Detener ejecución
    raise FileNotFoundError("Esperando archivo CSV de respuestas reales")
else:
    print("✅ Archivo encontrado. Procesando respuestas reales...")
    df_raw = pd.read_csv(ruta_csv)
    print(f"📊 Respuestas recibidas: {len(df_raw)}")
    print(f"📋 Columnas en el CSV: {len(df_raw.columns)}")

✅ Archivo encontrado. Procesando respuestas reales...
📊 Respuestas recibidas: 1
📋 Columnas en el CSV: 29


In [4]:

# MAPEO DE COLUMNAS: Google Forms → Preguntas q*


def encontrar_columna(df, texto_busqueda):
    """Busca una columna que contenga el texto especificado"""
    for col in df.columns:
        if texto_busqueda.lower() in str(col).lower():
            return col
    return None

# Mapeo de las 23 preguntas de EBLET-Lite
mapeo_preguntas = {
    # Burnout (4 preguntas) - Screening de agotamiento y desconexión
    'q1': 'emocionalmente agotado',
    'q2': 'verdadero esfuerzo',
    'q3': 'actitud distante',
    'q4': 'menos entusiasta',
    
    # Boreout (4 preguntas)
    'q5': 'monótono y repetitivo',
    'q6': 'carece de sentido',
    'q7': 'aburro en el trabajo',
    'q8': 'infrautilizado',
    
    # Bienestar (4 preguntas) - Adaptación WHO-5
    'q9': 'alegre y de buen humor',
    'q10': 'tranquilo',
    'q11': 'activo',
    'q12': 'energía',
    
    # Rotación (3 preguntas)
    'q13': 'buscando activamente',
    'q14': 'cambie de empresa',
    'q15': 'dejar mi trabajo',
    
    # Cultura CVF (8 preguntas) - Mantenido completo
    'q16': 'experimentar con nuevas ideas',
    'q17': 'creatividad y la innovación',
    'q18': 'colaboración entre compañeros',
    'q19': 'responsable se preocupa',
    'q20': 'objetivos son prioritarios',
    'q21': 'presión por conseguir',
    'q22': 'procedimientos están claramente',
    'q23': 'normas y reglas'
}

print("📋 Mapeo de 23 preguntas configurado:")
for q, texto in mapeo_preguntas.items():
    print(f"   {q}: '{texto}'")

📋 Mapeo de 23 preguntas configurado:
   q1: 'emocionalmente agotado'
   q2: 'verdadero esfuerzo'
   q3: 'actitud distante'
   q4: 'menos entusiasta'
   q5: 'monótono y repetitivo'
   q6: 'carece de sentido'
   q7: 'aburro en el trabajo'
   q8: 'infrautilizado'
   q9: 'alegre y de buen humor'
   q10: 'tranquilo'
   q11: 'activo'
   q12: 'energía'
   q13: 'buscando activamente'
   q14: 'cambie de empresa'
   q15: 'dejar mi trabajo'
   q16: 'experimentar con nuevas ideas'
   q17: 'creatividad y la innovación'
   q18: 'colaboración entre compañeros'
   q19: 'responsable se preocupa'
   q20: 'objetivos son prioritarios'
   q21: 'presión por conseguir'
   q22: 'procedimientos están claramente'
   q23: 'normas y reglas'


In [5]:

# FUNCIÓN PARA CALCULAR KPIs DE EBLET-LITE


def calcular_kpis_lite(respuestas):
    """
    Calcula KPIs a partir de las 23 respuestas de EBLET-Lite.
    
    Fórmulas:
    - Burnout Lite: (Q1+Q2+Q3+Q4)/4
    - Boreout Lite: (Q5+Q6+Q7+Q8)/4
    - Bienestar Lite: (Q9+Q10+Q11+Q12)/4
    - Rotación Lite: (Q13+Q14+Q15)/3
    - Cultura: Media de 2 preguntas por dimensión
    """
    if isinstance(respuestas, pd.DataFrame):
        if len(respuestas) == 1:
            respuestas = respuestas.iloc[0].to_dict()
        else:
            raise ValueError("Solo se admite una fila")
    
    kpis = {}
    
    # Burnout Lite (4 preguntas) - Screening de agotamiento y desconexión
    kpis["burnout"] = np.mean([
        respuestas.get('q1', 3),
        respuestas.get('q2', 3),
        respuestas.get('q3', 3),
        respuestas.get('q4', 3)
    ])
    
    # Boreout Lite (4 preguntas)
    kpis["boreout"] = np.mean([
        respuestas.get('q5', 3),
        respuestas.get('q6', 3),
        respuestas.get('q7', 3),
        respuestas.get('q8', 3)
    ])
    
    # Bienestar Lite (4 preguntas) - Adaptación WHO-5
    kpis["bienestar"] = np.mean([
        respuestas.get('q9', 3),
        respuestas.get('q10', 3),
        respuestas.get('q11', 3),
        respuestas.get('q12', 3)
    ])
    
    # Rotación Lite (3 preguntas)
    kpis["rotacion"] = np.mean([
        respuestas.get('q13', 3),
        respuestas.get('q14', 3),
        respuestas.get('q15', 3)
    ])
    
    # Cultura CVF (8 preguntas) - Mantenido completo de Enterprise
    kpis["cultura_adhocracia"] = np.mean([respuestas.get('q16', 3), respuestas.get('q17', 3)])
    kpis["cultura_clan"] = np.mean([respuestas.get('q18', 3), respuestas.get('q19', 3)])
    kpis["cultura_mercado"] = np.mean([respuestas.get('q20', 3), respuestas.get('q21', 3)])
    kpis["cultura_jerarquica"] = np.mean([respuestas.get('q22', 3), respuestas.get('q23', 3)])
    
    culturas = {
        "Adhocracia": kpis["cultura_adhocracia"],
        "Clan": kpis["cultura_clan"],
        "Mercado": kpis["cultura_mercado"],
        "Jerarquica": kpis["cultura_jerarquica"]
    }
    kpis["cultura_dominante"] = max(culturas, key=culturas.get)
    kpis["cultura_scores"] = culturas
    
    return kpis

print("✅ Función calcular_kpis_lite definida")
print("📊 Fórmulas:")
print("   - Burnout Lite: (Q1+Q2+Q3+Q4)/4")
print("   - Boreout Lite: (Q5+Q6+Q7+Q8)/4")
print("   - Bienestar Lite: (Q9+Q10+Q11+Q12)/4")
print("   - Rotación Lite: (Q13+Q14+Q15)/3")
print("   - Cultura: Media de 2 preguntas por dimensión")

✅ Función calcular_kpis_lite definida
📊 Fórmulas:
   - Burnout Lite: (Q1+Q2+Q3+Q4)/4
   - Boreout Lite: (Q5+Q6+Q7+Q8)/4
   - Bienestar Lite: (Q9+Q10+Q11+Q12)/4
   - Rotación Lite: (Q13+Q14+Q15)/3
   - Cultura: Media de 2 preguntas por dimensión


In [6]:

# CLASIFICADOR DE PERFILES PARA EBLET-LITE


def clasificar_individuo_lite(kpis):
    """
    Clasifica a una persona en uno de los 6 perfiles de bienestar.
    
    Nota: Esta clasificación se basa en indicadores de screening,
    no en medición completa de burnout (MBI-GS).
    """
    
    # Benchmark de referencia (12,500 empleados)
    BENCHMARK = {
        "burnout": {"p25": 2.1, "p50": 2.8, "p75": 3.6},
        "boreout": {"p25": 1.9, "p50": 2.5, "p75": 3.3},
        "bienestar": {"p25": 2.4, "p50": 3.1, "p75": 3.8}
    }
    
    def calcular_percentil(valor, kpi):
        bench = BENCHMARK[kpi]
        if valor <= bench["p25"]:
            return int((valor / bench["p25"]) * 25)
        elif valor <= bench["p50"]:
            return 25 + int(((valor - bench["p25"]) / (bench["p50"] - bench["p25"])) * 25)
        elif valor <= bench["p75"]:
            return 50 + int(((valor - bench["p50"]) / (bench["p75"] - bench["p50"])) * 25)
        else:
            return 75 + int(((valor - bench["p75"]) / (5 - bench["p75"])) * 25)
    
    percentiles = {
        "burnout": calcular_percentil(kpis["burnout"], "burnout"),
        "boreout": calcular_percentil(kpis["boreout"], "boreout"),
        "bienestar": calcular_percentil(kpis["bienestar"], "bienestar")
    }
    
    # Clasificación basada en umbrales
    if kpis["burnout"] < 2.5 and kpis["boreout"] < 2.5 and kpis["bienestar"] >= 3.5:
        perfil = "flourishing"
        nombre = " Flourishing"
        descripcion = "Indicadores de bienestar óptimo. Rendimiento y equilibrio adecuados."
    elif kpis["burnout"] >= 3.5 and kpis["boreout"] >= 3.5:
        perfil = "critico"
        nombre = " Crítico Dual"
        descripcion = "Indicadores elevados de agotamiento y desconexión. Requiere atención prioritaria."
    elif kpis["burnout"] >= 3.5:
        perfil = "quemado"
        nombre = "🟠 Quemado Activo"
        descripcion = "Indicadores de agotamiento elevados. Carga de trabajo y falta de recuperación."
    elif kpis["boreout"] >= 3.5:
        perfil = "aburrido"
        nombre = "🔵 Aburrido Crónico"
        descripcion = "Indicadores de desconexión elevados. Falta de estimulación y retos."
    elif kpis["rotacion"] >= 3.5 and kpis["bienestar"] < 3.0:
        perfil = "vuelo"
        nombre = "⚫ Desvinculado"
        descripcion = "Intención de cambio laboral elevada con bienestar bajo."
    else:
        perfil = "estable"
        nombre = "🟡 Estable Neutro"
        descripcion = "Indicadores en zona intermedia. Espacio de mejora."
    
    return {
        "perfil": perfil,
        "nombre": nombre,
        "descripcion": descripcion,
        "kpis": kpis,
        "percentiles": percentiles
    }

print("✅ Función clasificar_individuo_lite definida")
print("⚠️  Nota: Clasificación basada en screening, no en medición completa MBI-GS")

✅ Función clasificar_individuo_lite definida
⚠️  Nota: Clasificación basada en screening, no en medición completa MBI-GS


In [8]:

# PROCESAR RESPUESTAS REALES


print("🔧 Procesando respuestas reales del CSV...")

# Encontrar columna de alias
col_alias = encontrar_columna(df_raw, 'alias')

if col_alias is None:
    print("⚠️  No se encontró la columna de alias. Usando 'Respondiente_1', etc.")
    aliases = [f"Respondiente_{i+1}" for i in range(len(df_raw))]
else:
    aliases = df_raw[col_alias].fillna('Anónimo').tolist()

# Construir DataFrame de respuestas mapeadas
resultados_reales = []

for idx, row in df_raw.iterrows():
    resp = {'alias': aliases[idx]}
    
    for q_num, texto_busqueda in mapeo_preguntas.items():
        col_encontrada = encontrar_columna(df_raw, texto_busqueda)
        
        if col_encontrada:
            valor = row[col_encontrada]
            try:
                valor_num = int(valor)
                if 1 <= valor_num <= 5:
                    resp[q_num] = valor_num
                else:
                    resp[q_num] = 3
            except:
                resp[q_num] = 3
        else:
            resp[q_num] = 3
    
    # Calcular KPIs y clasificar
    kpis = calcular_kpis_lite(resp)
    perfil = clasificar_individuo_lite(kpis)
    
    resultados_reales.append({
        'alias': aliases[idx],
        'perfil': perfil['perfil'],
        'perfil_nombre': perfil['nombre'],
        'burnout': round(kpis['burnout'], 2),
        'boreout': round(kpis['boreout'], 2),
        'bienestar': round(kpis['bienestar'], 2),
        'rotacion': round(kpis['rotacion'], 2),
        'cultura_dominante': kpis.get('cultura_dominante', 'N/A'),
        'percentil_burnout': perfil['percentiles'].get('burnout', 50),
        'percentil_boreout': perfil['percentiles'].get('boreout', 50),
        'percentil_bienestar': perfil['percentiles'].get('bienestar', 50),
        '_kpis': kpis,
        '_perfil': perfil
    })

df_reales = pd.DataFrame(resultados_reales)

print(f"\n✅ {len(df_reales)} respuestas procesadas correctamente")

# Mostrar resumen

print("📊 RESUMEN DE PARTICIPANTES")
print(df_reales[['alias', 'perfil_nombre', 'cultura_dominante', 'burnout', 'boreout', 'bienestar']].to_string(index=False))

🔧 Procesando respuestas reales del CSV...

✅ 1 respuestas procesadas correctamente
📊 RESUMEN DE PARTICIPANTES
 alias perfil_nombre cultura_dominante  burnout  boreout  bienestar
Germán   Flourishing        Adhocracia      1.5      1.0        4.0


In [9]:

# DISTRIBUCIÓN DE PERFILES


distribucion = df_reales['perfil_nombre'].value_counts()

fig = px.pie(
    distribucion,
    values=distribucion.values,
    names=distribucion.index,
    title=f"📊 Distribución de Perfiles ({len(df_reales)} participantes reales)",
    color_discrete_map=color_perfil,
    hole=0.3
)

fig.update_layout(height=500, width=700)
fig.show()

print("\n📋 Distribución detallada:")
for perfil, count in distribucion.items():
    pct = count / len(df_reales) * 100
    print(f"   {perfil:25s}: {count:2d} ({pct:4.1f}%)")


📋 Distribución detallada:
    Flourishing             :  1 (100.0%)


In [11]:

# MAPA DE POSICIONAMIENTO CON ALIAS


fig = px.scatter(
    df_reales,
    x='burnout',
    y='boreout',
    color='perfil_nombre',
    color_discrete_map=color_perfil,
    hover_data=['alias', 'bienestar', 'cultura_dominante'],
    title="🗺️ Mapa de Posicionamiento: Agotamiento vs Desconexión",
    labels={'burnout': 'Agotamiento →', 'boreout': 'Desconexión →'},
    size_max=15
)

# Añadir etiquetas de alias
for idx, row in df_reales.iterrows():
    fig.add_annotation(
        x=row['burnout'],
        y=row['boreout'],
        text=row['alias'],
        showarrow=False,
        font=dict(size=9, color='black'),
        xshift=10,
        yshift=10
    )

# Líneas de umbral
fig.add_hline(y=3.5, line_dash="dash", line_color="red", opacity=0.4)
fig.add_vline(x=3.5, line_dash="dash", line_color="red", opacity=0.4)

# Etiquetas de cuadrantes
fig.add_annotation(x=1.5, y=1.5, text="🟢 Flourishing", showarrow=False, 
                   font_size=11, font=dict(color="#2ecc71", family="Arial Black"))
fig.add_annotation(x=4.5, y=1.5, text="🟠 Agotamiento", showarrow=False, 
                   font_size=11, font=dict(color="#e67e22", family="Arial Black"))
fig.add_annotation(x=1.5, y=4.5, text="🔵 Desconexión", showarrow=False, 
                   font_size=11, font=dict(color="#3498db", family="Arial Black"))
fig.add_annotation(x=4.5, y=4.5, text="🔴 Crítico", showarrow=False, 
                   font_size=11, font=dict(color="#e74c3c", family="Arial Black"))

fig.update_layout(
    height=650,
    width=900,
    xaxis_range=[1, 5.2],
    yaxis_range=[1, 5.2],
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5)
)

fig.show()

print("""
💡 INTERPRETACIÓN:
- Cada punto es un participante real alias visible 
- COLOR = perfil de bienestar
- Los 4 cuadrantes representan los estados organizacionales

""")


💡 INTERPRETACIÓN:
- Cada punto es un participante real alias visible 
- COLOR = perfil de bienestar
- Los 4 cuadrantes representan los estados organizacionales




In [12]:

# CULTURA PERCIBIDA POR PARTICIPANTE


culturas_data = []
for idx, row in df_reales.iterrows():
    kpis = row['_kpis']
    if 'cultura_scores' in kpis:
        culturas_data.append({
            'alias': row['alias'],
            'perfil': row['perfil_nombre'],
            'Adhocracia': kpis['cultura_scores']['Adhocracia'],
            'Clan': kpis['cultura_scores']['Clan'],
            'Mercado': kpis['cultura_scores']['Mercado'],
            'Jerarquica': kpis['cultura_scores']['Jerarquica']
        })

df_culturas = pd.DataFrame(culturas_data)

# Heatmap
fig = px.imshow(
    df_culturas[['Adhocracia', 'Clan', 'Mercado', 'Jerarquica']].values,
    x=['🔵 Adhocracia', '🟢 Clan', '🟠 Mercado', '🟡 Jerarquica'],
    y=[f"{r['alias']} ({r['perfil']})" for _, r in df_culturas.iterrows()],
    color_continuous_scale='Blues',
    zmin=1, zmax=5,
    title="🏛️ Cultura Percibida por Participante",
    text_auto='.1f',
    aspect='auto'
)

fig.update_layout(height=600, width=800)
fig.show()

print("\n📊 Distribución de culturas dominantes:")
print(df_reales['cultura_dominante'].value_counts().to_string())


📊 Distribución de culturas dominantes:
cultura_dominante
Adhocracia    1


In [13]:

# PERCENTILES VS BENCHMARK


fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        " Agotamiento - Percentiles",
        "😴 Desconexión - Percentiles",
        "💚 Bienestar - Percentiles"
    )
)

kpis_a_mostrar = ["burnout", "boreout", "bienestar"]
colores_kpi = ["#e74c3c", "#3498db", "#2ecc71"]

for i, (kpi, color) in enumerate(zip(kpis_a_mostrar, colores_kpi), 1):
    aliases = df_reales['alias'].tolist()
    percentiles = df_reales[f'percentil_{kpi}'].tolist()
    
    fig.add_trace(
        go.Bar(
            x=aliases,
            y=percentiles,
            marker_color=color,
            text=[f"P{p}" for p in percentiles],
            textposition='outside',
            name=kpi,
            showlegend=False
        ),
        row=1, col=i
    )
    
    fig.add_hline(y=25, line_dash="dot", line_color="green", opacity=0.5, row=1, col=i)
    fig.add_hline(y=50, line_dash="dot", line_color="yellow", opacity=0.5, row=1, col=i)
    fig.add_hline(y=75, line_dash="dot", line_color="red", opacity=0.5, row=1, col=i)

fig.update_layout(
    title_text="<b>📈 Posición de cada participante vs Benchmark (12,500 empleados)</b>",
    height=500,
    width=1200
)

for i in range(1, 4):
    fig.update_yaxes(range=[0, 100], row=1, col=i)

fig.show()

print("""
💡 INTERPRETACIÓN:
- Percentil 0-25: BAJO (mejor que el 75% del benchmark)
- Percentil 25-50: MEDIO-BAJO
- Percentil 50-75: MEDIO-ALTO
- Percentil 75-100: ALTO (peor que el 75% del benchmark)

⚠️ Para Agotamiento/Desconexión: percentil ALTO = MALO
✅ Para Bienestar: percentil ALTO = BUENO
""")


💡 INTERPRETACIÓN:
- Percentil 0-25: BAJO (mejor que el 75% del benchmark)
- Percentil 25-50: MEDIO-BAJO
- Percentil 50-75: MEDIO-ALTO
- Percentil 75-100: ALTO (peor que el 75% del benchmark)

⚠️ Para Agotamiento/Desconexión: percentil ALTO = MALO
✅ Para Bienestar: percentil ALTO = BUENO



In [14]:

# RESUMEN FINAL Y LIMITACIONES


print("""

                🎯 EBLET-LITE - RESUMEN DE RESULTADOS                    

                                                                              
  📊 PARTICIPANTES PROCESADOS: """ + f"{len(df_reales)}".ljust(62) + """
                                                                              
  📋 DISTRIBUCIÓN DE PERFILES:                                                
""")

for perfil, count in df_reales['perfil_nombre'].value_counts().items():
    pct = count / len(df_reales) * 100
    print(f"║    {perfil:25s}: {count:2d} ({pct:4.1f}%)".ljust(80) + "║")

print("""                                                                             
  🏛️ CULTURAS DOMINANTES:                                                     
""")

for cultura, count in df_reales['cultura_dominante'].value_counts().items():
    pct = count / len(df_reales) * 100
    print(f"    {cultura:25s}: {count:2d} ({pct:4.1f}%)".ljust(80) + "")


print("\n✅ NOTEBOOK EBLET-LITE COMPLETADO")
print("📋 Datos reales procesados: " + str(len(df_reales)) + " participantes")



                🎯 EBLET-LITE - RESUMEN DE RESULTADOS                    


  📊 PARTICIPANTES PROCESADOS: 1                                                             

  📋 DISTRIBUCIÓN DE PERFILES:                                                

║     Flourishing             :  1 (100.0%)                                     ║
                                                                             
  🏛️ CULTURAS DOMINANTES:                                                     

    Adhocracia               :  1 (100.0%)                                      

✅ NOTEBOOK EBLET-LITE COMPLETADO
📋 Datos reales procesados: 1 participantes
